# 数据类型转换：日期 / 金额 / 类别，彻底搞定格式问题

> 这是「数据分析从入门到精通」系列的第 12 篇。数据不缺了，但类型不对同样会让后续分析出错——日期是字符串、金额带逗号、类别列占内存……这篇一次解决。

---

嗨，我是小荷。

我见过太多人在数据清洗上卡住，不是因为缺失值，而是因为**类型问题**。

比如你想按日期筛选数据，结果发现 `日期` 列的 dtype 是 `object`（字符串），直接用 `> '2024-01-01'` 比较出来的结果完全不对；或者 `金额` 列里有逗号分隔符，`8,500` 这个值 Python 认不出来，根本没法做加法。

这种问题萧何见了也得皱眉头——账本上的日期写的是"正月初一"，你让计算机算几月几号？**数据格式不对，后面什么都白搭**。

---

## 查看和转换数据类型

In [30]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    '订单日期': ['2024-01-03', '2024-01-05', '2024-02-10'],
    '金额': ['8,500', '1,200.50', '3,600'],
    '数量': ['3', '1', '5'],
    '用户等级': ['普通', 'VIP', '普通'],
    '是否退款': ['True', 'False', 'True']
})

# 查看当前类型
print(df.dtypes)


订单日期    str
金额      str
数量      str
用户等级    str
是否退款    str
dtype: object


---

## 日期类型转换

### pd.to_datetime()

In [16]:
# 基础转换
df['订单日期'] = pd.to_datetime(df['订单日期'])
print(df['订单日期'].dtype)  # datetime64[ns]

# 指定格式（非标准格式必须指定，否则可能解析错误）
pd.to_datetime('20240103', format='%Y%m%d')

datetime64[us]


Timestamp('2024-01-03 00:00:00')

In [17]:
pd.to_datetime('03/01/2024', format='%d/%m/%Y')

Timestamp('2024-01-03 00:00:00')

In [18]:
pd.to_datetime('2024年1月3日', format='%Y年%m月%d日')

Timestamp('2024-01-03 00:00:00')

In [19]:
# errors='coerce'：无法解析的值变为 NaT（Not a Time），不报错
df['订单日期'] = pd.to_datetime(df['订单日期'], errors='coerce')
df

,订单日期,金额,数量,用户等级,是否退款
0,2024-01-03,"8,500",3,普通,True
1,2024-01-05,"1,200.50",1,VIP,False
2,2024-02-10,"3,600",5,普通,True


### 日期列解锁的功能

转成 datetime 之后，就可以用 `.dt` 访问器做很多操作：

In [20]:
df['订单日期'] = pd.to_datetime(df['订单日期'])

df['年'] = df['订单日期'].dt.year
df['月'] = df['订单日期'].dt.month
df['日'] = df['订单日期'].dt.day
df['星期几'] = df['订单日期'].dt.day_name()     # 'Monday'...
df['是否周末'] = df['订单日期'].dt.dayofweek >= 5
df['季度'] = df['订单日期'].dt.quarter

# 时间差计算
today = pd.Timestamp('2024-03-01')
df['距今天数'] = (today - df['订单日期']).dt.days

df

,订单日期,金额,数量,用户等级,是否退款,年,月,日,星期几,是否周末,季度,距今天数
0,2024-01-03,"8,500",3,普通,True,2024,1,3,Wednesday,False,1,58
1,2024-01-05,"1,200.50",1,VIP,False,2024,1,5,Friday,False,1,56
2,2024-02-10,"3,600",5,普通,True,2024,2,10,Saturday,True,1,20


In [21]:
# 按月份筛选
jan_orders = df[df['订单日期'].dt.month == 1]
jan_orders

,订单日期,金额,数量,用户等级,是否退款,年,月,日,星期几,是否周末,季度,距今天数
0,2024-01-03,"8,500",3,普通,True,2024,1,3,Wednesday,False,1,58
1,2024-01-05,"1,200.50",1,VIP,False,2024,1,5,Friday,False,1,56


---

## 数值类型转换

### pd.to_numeric()

In [27]:
# 去掉逗号，转成数值
df['金额']=pd.Series(['8,500', '1,200.50', '3,600'])
df['金额'] = df['金额'].str.replace(',', '').astype(float)
df['金额'].dtypes

dtype('float64')

In [25]:
# pd.to_numeric（推荐，可以处理错误）
df['金额']=pd.Series(['8,500', '1,200.50', '3,600'])
df['金额'] = pd.to_numeric(df['金额'].str.replace(',', ''), errors='coerce')
df['金额'].dtypes

dtype('float64')

In [31]:
# 整数转换
print('转换前类型： ', df['数量'].dtypes)
df['数量'] = df['数量'].astype(int)
print('转换后类型： ', df['数量'].dtypes)

转换前类型：  str
转换后类型：  int64


### 去除金额里的特殊字符

In [32]:
# 金额列可能含 ¥、$、空格等
prices = pd.Series(['¥8,500', '$1,200.50', '3,600元', ' 500 '])

# 链式清洗
prices_clean = prices\
    .str.strip()           \
    .str.replace('¥', '')  \
    .str.replace('$', '')  \
    .str.replace('元', '')  \
    .str.replace(',', '')  \
    .astype(float)

print(prices_clean)
# 0    8500.00
# 1    1200.50
# 2    3600.00
# 3     500.00


0    8500.0
1    1200.5
2    3600.0
3     500.0
dtype: float64


---

## 布尔类型转换

In [37]:
# 字符串 'True'/'False' → 真正的布尔值
df['是否退款'] = pd.Series(['True', 'False', 'True'])
df['是否退款'] = df['是否退款'].map({'True': True, 'False': False})
df['是否退款'] 

0     True
1    False
2     True
Name: 是否退款, dtype: bool

In [42]:
# 或者
df['是否退款'] = pd.Series(['True', 'False', 'True'])
df['是否退款'] = df['是否退款'] == 'True'
df['是否退款'] 

0     True
1    False
2     True
Name: 是否退款, dtype: bool

In [43]:
# 1/0 → 布尔
df['是否退款'] = df['是否退款'].astype(bool)
df['是否退款'] 

0     True
1    False
2     True
Name: 是否退款, dtype: bool

---

## 类别类型（Category）：省内存的神器

对于取值数量有限的列（如城市级别、性别、等级），转成 `category` 类型能大幅减少内存占用：

In [45]:
df = pd.read_csv('全国城市数据.csv', encoding='gbk')

# 转换前
print(f"城市级别 - 转换前: {df['城市级别'].memory_usage()} bytes, dtype: {df['城市级别'].dtype}")

# 转换后
df['城市级别'] = df['城市级别'].astype('category')
print(f"城市级别 - 转换后: {df['城市级别'].memory_usage()} bytes, dtype: {df['城市级别'].dtype}")


城市级别 - 转换前: 2876 bytes, dtype: str
城市级别 - 转换后: 515 bytes, dtype: category


**内存节省效果：**
```
object 类型：每个字符串都单独存储
category 类型：底层用整数编码，只存一份字符串映射

数据量越大，效果越明显。百万行数据能省几百 MB。
```

类别类型还可以指定顺序（有序类别）：

In [46]:
from pandas.api.types import CategoricalDtype

# 定义有序类别
level_order = CategoricalDtype(
    categories=['其他', '三线', '二线', '准一线', '一线'],
    ordered=True
)
df['城市级别'] = df['城市级别'].astype(level_order)

# 现在可以直接比较大小
big = df[df['城市级别'] >= '二线']
print(big['城市级别'].value_counts())


城市级别
二线     170
准一线     37
一线       4
其他       0
三线       0
Name: count, dtype: int64


---

## 实战：清洗一份格式混乱的数据表

In [47]:
import pandas as pd

# 模拟一份从 Excel 导出的混乱数据
raw = pd.DataFrame({
    '订单号': ['ORD001', 'ORD002', 'ORD003', 'ORD004'],
    '下单时间': ['2024/01/03 10:25', '2024-1-5', '20240210', '2024.03.01'],
    '金额(元)': ['¥1,299', '899.00', '2,580元', '¥450'],
    '数量': ['2', '1', '3', '1'],
    '客户等级': ['普通', 'VIP', 'SVIP', '普通'],
    '已付款': ['是', '是', '否', '是']
})

df = raw.copy()

# 1. 日期列：统一解析
df['下单时间'] = pd.to_datetime(df['下单时间'].str.replace('.', '-').str.replace('/', '-'),
                                format='mixed', errors='coerce')

# 2. 金额列：去除特殊符号，转数值
df['金额(元)'] = pd.to_numeric(
    df['金额(元)'].str.replace('[¥,元]', '', regex=True),
    errors='coerce'
)

# 3. 数量：转整数
df['数量'] = df['数量'].astype(int)

# 4. 客户等级：转有序类别
cat_type = pd.CategoricalDtype(['普通', 'VIP', 'SVIP'], ordered=True)
df['客户等级'] = df['客户等级'].astype(cat_type)

# 5. 付款状态：转布尔
df['已付款'] = df['已付款'].map({'是': True, '否': False})

print(df.dtypes)
print("\n清洗后数据:")
print(df)


订单号                 str
下单时间     datetime64[us]
金额(元)           float64
数量                int64
客户等级           category
已付款                bool
dtype: object

清洗后数据:
      订单号                下单时间   金额(元)  数量  客户等级    已付款
0  ORD001 2024-01-03 10:25:00  1299.0   2    普通   True
1  ORD002 2024-01-05 00:00:00   899.0   1   VIP   True
2  ORD003 2024-02-10 00:00:00  2580.0   3  SVIP  False
3  ORD004 2024-03-01 00:00:00   450.0   1    普通   True


---

## 本篇小结

| 目标类型 | 函数 | 注意事项 |
|---------|------|---------|
| datetime | `pd.to_datetime()` | 非标准格式指定 format |
| float/int | `astype(float)` / `pd.to_numeric()` | 先清除特殊字符 |
| bool | `astype(bool)` / `map()` | 注意字符串"True"≠True |
| category | `astype('category')` | 有序类别用 CategoricalDtype |

---

## 课后练习

In [55]:
import pandas as pd

df = pd.DataFrame({
    '注册时间': ['2023年1月5日', '2023年3月12日', '2022年11月8日','2022年12月8日'],
    '充值金额': ['100.00元', '￥588', '1,200','99.00元'],
    '会员等级': ['青铜', '白银', '黄金','青铜'],
    '活跃': ['yes', 'no', 'yes','yes']
})

# 任务1：注册时间 → datetime 类型
# 任务2：充值金额 → float 类型
# 任务3：会员等级 → 有序 category（青铜 < 白银 < 黄金）
# 任务4：活跃 → bool 类型


In [56]:
# 任务1：注册时间 → datetime
df['注册时间'] = pd.to_datetime(df['注册时间'].str.replace('日', ''), format='%Y年%m月%d')

In [57]:
# 任务2：充值金额 → float
df['充值金额'] = (df['充值金额']
                 .str.replace('元|￥|,', '', regex=True)  # 去符号
                 .astype(float))

In [58]:
# 任务3：会员等级 → 有序 category
level_order = ['青铜', '白银', '黄金']
df['会员等级'] = pd.Categorical(df['会员等级'], categories=level_order, ordered=True)

In [59]:
# 任务4：活跃 → bool
df['活跃'] = df['活跃'].map({'yes': True, 'no': False})

# 验证
print(df.dtypes)
print(df)

注册时间    datetime64[us]
充值金额           float64
会员等级          category
活跃                bool
dtype: object
        注册时间    充值金额 会员等级     活跃
0 2023-01-05   100.0   青铜   True
1 2023-03-12   588.0   白银  False
2 2022-11-08  1200.0   黄金   True
3 2022-12-08    99.0   青铜   True


评论区见 👀

本篇完整代码包括练习题解答都已经上传至 GitHub 仓库，欢迎 Clone。

---

## 下期预告

> **第 13 篇：字符串处理与正则表达式**
>
> 日期和数字搞定了，字符串呢？手机号、地址、姓名这些文本数据怎么清洗？Pandas 的 str 系列函数 + 正则，下篇全搞定。

---

👇 点「在看」，推给每天和脏数据作斗争的同学  
💬 评论区说说你遇到过最奇葩的日期格式  
⭐ 关注公众号，跟萧何迷妹学干净数据处理

---

*「数据分析从入门到精通」系列 · 第 12 篇*  
*上一篇：[第 11 篇：缺失值处理]*  
*下一篇：第 13 篇：字符串处理与正则表达式*